In [3]:
# @title 🚀 Install ComfyUI {"form-width":"30%"}
from pathlib import Path
from IPython.display import display, HTML, Javascript

OPTIONS = {}
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True # @param {"type":"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial ComfyUI setup =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Installing dependencies =-
!pip install xformers -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu126

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes
  ### Install custom nodes
  !git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack comfyui-impact-pack
  !git clone https://github.com/ltdrdata/ComfyUI-Impact-Subpack
  !git clone https://github.com/rgthree/rgthree-comfy.git
  !git clone https://github.com/MoonGoblinDev/Civicomfy.git
  !git clone https://github.com/calcuis/gguf.git

  # Correction of permissions
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat
  ![ ! -d ComfyUI-Manager ] && echo -= Initial ComfyUI-Manager setup =- && git clone https://github.com/ltdrdata/ComfyUI-Manager

  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Installing custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

from IPython.display import clear_output, display, HTML
clear_output()

display(HTML("""
<div style="text-align: center; margin-top: 20px;">
    <button style="
        background-color: #4CAF50;
        border: none;
        color: white;
        padding: 15px 32px;
        text-align: center;
        text-decoration: none;
        display: inline-block;
        font-size: 16px;
        margin: 4px 2px;
        cursor: not-allowed;
        border-radius: 8px;
        font-weight: bold;
        opacity: 0.9;
    ">
        ✓ Done
    </button>
    <p style="color: #4CAF50; font-weight: bold; margin-top: 10px;">
        ✅ ComfyUI installed successfully!
    </p>
</div>
"""))

In [3]:
# @title 📥 Download Models {"form-width":"30%"}
from IPython.display import display, HTML, Javascript
import ipywidgets as widgets
from IPython.display import clear_output

# Configuration options
MODEL_VERSION = "Wan 2.2 (LowNoise)" #@param ["Wan 2.2 (LowNoise)", "Wan 2.1"]
CUSTOM_LORA_URL = "" #@param {type:"string"}

# Install aria2
!apt-get update -qq
!apt-get install -y aria2

# Define directories
diffusion_dir = "/content/ComfyUI/models/diffusion_models"
text_encoders_dir = "/content/ComfyUI/models/text_encoders"
vae_dir = "/content/ComfyUI/models/vae"
loras_dir = "/content/ComfyUI/models/loras"

# Create directories if they don't exist
!mkdir -p {diffusion_dir}
!mkdir -p {text_encoders_dir}
!mkdir -p {vae_dir}
!mkdir -p {loras_dir}

# Model configurations
models_config = {
    "Wan 2.2 (LowNoise)": {
        "diffusion": {
            "url": "https://huggingface.co/QuantStack/Wan2.2-T2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-T2V-A14B-LowNoise-Q4_K_M.gguf?download=true",
            "output": "wan2.2_t2v_14B_lownoise.gguf"
        },
        "text_encoder": {
            "url": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors?download=true",
            "output": "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
        },
        "vae": {
            "url": "https://huggingface.co/QuantStack/Wan2.2-T2V-A14B-GGUF/resolve/main/VAE/Wan2.1_VAE.safetensors?download=true",
            "output": "wan_2.2_vae.safetensors"
        }
    },
    "Wan 2.1": {
        "diffusion": {
            "url": "https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q4_K_M.gguf?download=true",
            "output": "wan2.1_t2v_14B_fp16.gguf"
        },
        "text_encoder": {
            "url": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors?download=true",
            "output": "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
        },
        "vae": {
            "url": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors?download=true",
            "output": "wan_2.1_vae.safetensors"
        }
    }
}

# Get selected configuration
selected_config = models_config[MODEL_VERSION]

print(f"📦 Selected model: {MODEL_VERSION}")
print("="*60)

# Download diffusion model
print("\n🔄 Downloading diffusion model...")
!aria2c -x 16 -s 16 -k 1M \
  "{selected_config['diffusion']['url']}" \
  -d {diffusion_dir} \
  -o "{selected_config['diffusion']['output']}"

# Download text encoder (shared between versions)
print("\n🔄 Downloading text encoder...")
print("ℹ️ Note: Text encoder is shared between Wan 2.1 and 2.2")
!aria2c -x 16 -s 16 -k 1M \
  "{selected_config['text_encoder']['url']}" \
  -d {text_encoders_dir} \
  -o "{selected_config['text_encoder']['output']}"

# Download VAE
print("\n🔄 Downloading VAE...")
!aria2c -x 16 -s 16 -k 1M \
  "{selected_config['vae']['url']}" \
  -d {vae_dir} \
  -o "{selected_config['vae']['output']}"

# Download custom LoRA if URL is provided
if CUSTOM_LORA_URL and CUSTOM_LORA_URL.strip():
    print("\n🔄 Downloading custom LoRA...")
    # Extract filename from URL or use default
    import os
    lora_filename = os.path.basename(CUSTOM_LORA_URL.split('?')[0])
    if not lora_filename.endswith('.safetensors'):
        lora_filename = "custom_lora.safetensors"

    !aria2c -x 16 -s 16 -k 1M \
      "{CUSTOM_LORA_URL}" \
      -d {loras_dir} \
      -o "{lora_filename}"
    print(f"✅ LoRA saved as: {lora_filename}")
else:
    print("\nℹ️ No custom LoRA URL provided - skipping LoRA download")
    lora_filename = None

print("\n" + "="*60)
print("✅ All files downloaded successfully!")
print("="*60)

# Clear all previous output
clear_output()

# Prepare LoRA info if applicable
lora_info = ""
if CUSTOM_LORA_URL and CUSTOM_LORA_URL.strip():
    import os
    if 'lora_filename' not in locals():
        lora_filename = os.path.basename(CUSTOM_LORA_URL.split('?')[0])
        if not lora_filename.endswith('.safetensors'):
            lora_filename = "custom_lora.safetensors"
    lora_info = f"<li style='margin: 10px 0;'><strong>LoRA:</strong> {lora_filename}</li>"

# Display summary with Done button
display(HTML(f"""
<div style="text-align: center; margin-top: 20px;">
    <div style="
        background-color: #f8f9fa;
        border: 2px solid #4CAF50;
        border-radius: 12px;
        padding: 20px;
        margin-bottom: 20px;
        max-width: 600px;
        margin-left: auto;
        margin-right: auto;
    ">
        <h2 style="color: #2c3e50; margin-top: 0;">📊 Download Summary</h2>
        <ul style="
            list-style: none;
            padding: 0;
            text-align: left;
            display: inline-block;
        ">
            <li style="margin: 10px 0;"><strong>Model Version:</strong> {MODEL_VERSION}</li>
            <li style="margin: 10px 0;"><strong>Diffusion Model:</strong> {selected_config['diffusion']['output']}</li>
            <li style="margin: 10px 0;"><strong>Text Encoder:</strong> {selected_config['text_encoder']['output']}</li>
            <li style="margin: 10px 0;"><strong>VAE:</strong> {selected_config['vae']['output']}</li>
            {lora_info}
        </ul>
    </div>

    <button style="
        background-color: #4CAF50;
        border: none;
        color: white;
        padding: 15px 32px;
        text-align: center;
        text-decoration: none;
        display: inline-block;
        font-size: 16px;
        margin: 4px 2px;
        cursor: not-allowed;
        border-radius: 8px;
        font-weight: bold;
        opacity: 0.9;
    ">
        ✓ Done
    </button>

    <p style="color: #4CAF50; font-weight: bold; margin-top: 10px;">
        ✅ All models downloaded successfully!
    </p>
</div>
"""))

In [ ]:
# @title 🎬 Launch ComfyUI{"form-width":"20%"}
from google.colab import output
import threading
import time
import socket
import os
import shutil
from pathlib import Path
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

SAVE_TO_DRIVE = True  #@param {type:"boolean"}

!pip install -q watchdog

if SAVE_TO_DRIVE:
    print("🔗 Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!\n")

!cd /content/ComfyUI

class VideoCopyHandler(FileSystemEventHandler):
    def __init__(self, source_dir, dest_dir):
        self.source_dir = source_dir
        self.dest_dir = dest_dir
        self.copied_files = set()
        self.pending_files = {}

    def on_created(self, event):
        if event.is_directory:
            return

        if event.src_path.lower().endswith(('.mp4', '.webm', '.avi', '.mov', '.png', '.jpg', '.jpeg', '.webp', '.gif')):
            self.pending_files[event.src_path] = time.time()

    def on_modified(self, event):
        if event.is_directory:
            return

        if event.src_path.lower().endswith(('.mp4', '.webm', '.avi', '.mov', '.png', '.jpg', '.jpeg', '.webp', '.gif')):
            self.pending_files[event.src_path] = time.time()

    def wait_for_file_complete(self, file_path, timeout=30):
        if not os.path.exists(file_path):
            return False

        last_size = -1
        stable_count = 0
        start_time = time.time()

        while time.time() - start_time < timeout:
            try:
                current_size = os.path.getsize(file_path)

                if current_size == 0:
                    time.sleep(0.5)
                    continue

                if current_size == last_size:
                    stable_count += 1
                    if stable_count >= 5:
                        return True
                else:
                    stable_count = 0

                last_size = current_size
                time.sleep(0.5)

            except Exception as e:
                time.sleep(0.5)
                continue

        try:
            final_size = os.path.getsize(file_path)
            return final_size > 0
        except:
            return False

    def copy_to_drive(self, file_path):
        if file_path in self.copied_files:
            return

        timeout = 60 if file_path.lower().endswith(('.mp4', '.webm', '.avi', '.mov')) else 15

        if not self.wait_for_file_complete(file_path, timeout=timeout):
            print(f"⚠️ File {os.path.basename(file_path)} not ready for copying")
            return

        try:
            file_size = os.path.getsize(file_path)
            if file_size < 1000:
                print(f"⚠️ File {os.path.basename(file_path)} too small ({file_size} bytes)")
                return

            rel_path = os.path.relpath(file_path, self.source_dir)
            dest_path = os.path.join(self.dest_dir, rel_path)

            os.makedirs(os.path.dirname(dest_path), exist_ok=True)

            print(f"📤 Copying: {os.path.basename(file_path)}...", end='', flush=True)
            shutil.copy2(file_path, dest_path)

            if os.path.exists(dest_path):
                dest_size = os.path.getsize(dest_path)
                if dest_size == file_size:
                    file_size_mb = file_size / (1024 * 1024)
                    filename = os.path.basename(file_path)
                    file_type = "🎬" if file_path.lower().endswith(('.mp4', '.webm', '.avi', '.mov')) else "🖼️"
                    print(f"\r{file_type} Saved to Drive: {filename} ({file_size_mb:.2f} MB)")
                    self.copied_files.add(file_path)
                else:
                    print(f"\r❌ Error: size mismatch")
                    os.remove(dest_path)
            else:
                print(f"\r❌ File did not appear on Drive")

        except Exception as e:
            print(f"\r❌ Error: {e}")

    def process_pending_files(self):
        current_time = time.time()
        files_to_copy = []

        for file_path, last_modified in list(self.pending_files.items()):
            wait_time = 5.0 if file_path.lower().endswith(('.mp4', '.webm', '.avi', '.mov')) else 2.0

            if current_time - last_modified > wait_time:
                files_to_copy.append(file_path)
                del self.pending_files[file_path]

        for file_path in files_to_copy:
            if file_path not in self.copied_files:
                self.copy_to_drive(file_path)

def start_video_watcher():
    if not SAVE_TO_DRIVE:
        print("ℹ️ Auto-copy to Drive disabled")
        return None

    source_dir = "/content/ComfyUI/output"
    dest_dir = "/content/drive/MyDrive/ComfyUI_Wan2.1_Output"

    if not os.path.exists("/content/drive/MyDrive"):
        print("⚠️ Google Drive not mounted! Files will not be copied.")
        return None

    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(source_dir):
        existing_files = []
        for root, dirs, files in os.walk(source_dir):
            for file in files:
                if file.lower().endswith(('.mp4', '.webm', '.avi', '.mov', '.png', '.jpg', '.jpeg', '.webp', '.gif')):
                    src_path = os.path.join(root, file)

                    if os.path.getsize(src_path) < 1000:
                        continue

                    rel_path = os.path.relpath(src_path, source_dir)
                    dest_path = os.path.join(dest_dir, rel_path)

                    if not os.path.exists(dest_path):
                        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
                        shutil.copy2(src_path, dest_path)

                        if os.path.getsize(dest_path) == os.path.getsize(src_path):
                            existing_files.append(file)

        if existing_files:
            print(f"📦 Copied {len(existing_files)} existing files to Drive")

    event_handler = VideoCopyHandler(source_dir, dest_dir)
    observer = Observer()
    observer.schedule(event_handler, source_dir, recursive=True)
    observer.start()

    def process_pending_loop():
        while True:
            time.sleep(3)
            event_handler.process_pending_files()

    threading.Thread(target=process_pending_loop, daemon=True).start()

    print("👀 Output folder monitoring started")
    print(f"📂 Local: {source_dir}")
    print(f"☁️ Google Drive: {dest_dir}")
    print("✅ New videos and images will auto-copy to Drive\n")

    return observer

from IPython.display import clear_output, display, HTML
clear_output()

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()

    print("\n" + "="*50)
    print("🎬 ComfyUI Wan2.1 ready to work!")
    print("="*50 + "\n")

    start_video_watcher()

    output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
!python /content/ComfyUI/main.py --dont-print-server --lowvram